# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
from datetime import datetime
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, RetrievalEvaluation
from lib.tooling import tool
from typing import List, Optional, Dict, Any


In [3]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

#CONSTANTS
load_dotenv('.env')

CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

CHROMADB_PATH = "chromadb"
COLLECTION_NAME = "udaplay"


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game



In [4]:
@tool
def retrieve_game(query:str, n_results:int):

    # Initialize Chroma and get the collection
    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=CHROMA_OPENAI_API_KEY
        )

    chroma_client = chromadb.PersistentClient(
        path=CHROMADB_PATH
        )

    collection = chroma_client.get_collection(
        COLLECTION_NAME, 
        embedding_function=embedding_fn
        )

    # Query the database

    results = collection.query(query_texts=[query], 
                               n_results=n_results, 
                               include=['documents'])
    retrieved_docs = results['documents'][0]
    return retrieved_docs

#### Evaluate Retrieval Tool

In [ ]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

In [5]:
@tool
def evaluate_retrieval(question:str, retrieved_docs:list) -> RetrievalEvaluation:
    
    #Initialize the Judge LLM
    judge_llm = LLM(
    model="gpt-4o-mini", 
    temperature=0.0,
    api_key=OPENAI_API_KEY
    )
    
    # Turn docs into a readable string for the judge LLM
    docs_text = "\n\n".join(f"- {doc}" for doc in retrieved_docs)

    #2. Build the Messages
    system_msg = SystemMessage(
        content=(
            "You are a strict evaluator for a retrieval-augmented generation (RAG) system. "
            "Given a question and a set of retrieved documents, your job is ONLY to decide "
            "whether these documents are sufficient to answer the question, and to explain why.\n\n"
            "You must respond using the structured schema you have been given."
        )
    )

    user_msg = UserMessage(
        content=(
            f"Question:\n{question}\\n"
            f"Retrieved docuemnts:\n{docs_text}\n\n"
            "Decide if these documents are enough to answer the question correctly."
        )
    )

    ai_message = judge_llm.invoke(
        input=[system_msg, user_msg],
        response_format=RetrievalEvaluation
    )

    evaluation: RetrievalEvaluation = ai_message.content
    return evaluation

#### Game Web Search Tool

In [ ]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 



In [20]:
@tool
def game_web_search(question:str) -> Dict:

    #Initialize the Tavily Client
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    
    # Perform the search
    search_result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }
    
    return formatted_results

### Agent

In [21]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [retrieve_game, evaluate_retrieval, game_web_search]

instructions = (
"You are a research agent for the video game industry."
"You can answer multistep questions by sequentially calling functions for:"
"1. Answer questions using internal knowledge (RAG)"
"2. Run the evaluate_retrieval tool to see if the retrieved information is enought to answer"
"2. Search the web when the internal knowledge is insufficient using the game web search"
"3. Maintain conversation state"
"4. Return structured outputs"
"You follow a pattern of of Thought and Action. "
"Create a plan of execution: "
"- Use Thought to describe your thoughts about the question you have been asked. "
"- Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly."
"When you think it's over, return the answer "
"Never try to respond directly if the question needs a tool. "
"But if you don't have a tool available, you can respond directly. "
f"The actions you have are the Tools: {tools}. \n"
)

agent = Agent(
    model_name="gpt-4o-mini",
    tools=tools,
    instructions=instructions,
)

In [22]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

q1 = "When Pokémon Gold and Silver was released?"
q2 = "Which one was the first 3D platformer Mario game?"
q3 = "Was Mortal Kombat X realeased for Playstation 5?"


In [23]:
run_object = agent.invoke(
    query=q1,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="You are a research agent for the video game industry.You can answer multistep questions by sequentially calling functions for:1. Answer questions using internal knowledge (RAG)2. Run the evaluate_retrieval tool to see if the retrieved information is enought to answer2. Search the web when the internal knowledge is insufficient using the game web search3. Maintain conversation state4. Return structured outputsYou follow a pattern of of Thought and Action. Create a plan of execution: - Use Thought to describe your thoughts about the question you have been asked. - Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly.When you think it's over, return the answer Never try to respond directly if the question needs a tool. But if you don't have a tool available, you can respond directly. The actions you have are the Tools: [<Tool name=retrieve_game params=['query', 'n_results']>, <Tool name

In [ ]:
run_object = agent.invoke(
    query=q2,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="You are a research agent for the video game industry.You can answer multistep questions by sequentially calling functions for:1. Answer questions using internal knowledge (RAG)2. Run the evaluate_retrieval tool to see if the retrieved information is enought to answer2. Search the web when the internal knowledge is insufficient using the game web search3. Maintain conversation state4. Return structured outputsYou follow a pattern of of Thought and Action. Create a plan of execution: - Use Thought to describe your thoughts about the question you have been asked. - Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly.When you think it's over, return the answer Never try to respond directly if the question needs a tool. But if you don't have a tool available, you can respond directly. The actions you have are the Tools: [<Tool name=retrieve_game params=['query', 'n_results']>, <Tool name

In [25]:
run_object = agent.invoke(
    query=q3,
)
run_object.get_final_state()["messages"]

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


[SystemMessage(role='system', content="You are a research agent for the video game industry.You can answer multistep questions by sequentially calling functions for:1. Answer questions using internal knowledge (RAG)2. Run the evaluate_retrieval tool to see if the retrieved information is enought to answer2. Search the web when the internal knowledge is insufficient using the game web search3. Maintain conversation state4. Return structured outputsYou follow a pattern of of Thought and Action. Create a plan of execution: - Use Thought to describe your thoughts about the question you have been asked. - Use Action to specify one of the tools available to you. if you don't have a tool available, you can respond directly.When you think it's over, return the answer Never try to respond directly if the question needs a tool. But if you don't have a tool available, you can respond directly. The actions you have are the Tools: [<Tool name=retrieve_game params=['query', 'n_results']>, <Tool name

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes